# BlinkMimic T4 全 40 段动作视频渲染

指标和轨迹录制已经完成；本 Notebook 只下载已保存的 `eval_traj.npz` 并渲染完整 40 段动作，不重新运行策略评测。

每段 3 秒、15 FPS、640×360。旧模型和新模型分别生成完整视频并上传到 HF。

In [ ]:
from pathlib import Path
import os

RUNS = {
    'old_083958': {'run_id': '20261122_083958'},
    'new_105913': {'run_id': '20261122_105913'},
}
REPO_DIR = Path('/kaggle/working/proj2-humanoid-mimic/third_party/blinkmimic')
REPO_URL = 'https://github.com/fuyichen0725/blinkmimic.git'
COMMIT = 'd264658d6c9ce61fb52f9f0f8985a369f3ccc409'
BASE_RESULTS = Path('/kaggle/working/proj2-humanoid-mimic/results/blinkmimic_t4_ddp')
HF_REPO_NAME = 'proj2-blinkmimic-t4'
CLIP_SECONDS = 3
VIDEO_FPS = 15
VIDEO_WIDTH = 640
VIDEO_HEIGHT = 360
HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN
except Exception as exc:
    print(f'kaggle_secrets unavailable: {exc}')
for item in RUNS.values():
    item['root'] = BASE_RESULTS / f"run_{item['run_id']}"
print('HF_TOKEN=' + ('set' if HF_TOKEN else 'MISSING'))
print({name: item['run_id'] for name, item in RUNS.items()})


In [ ]:
import subprocess
def run(command, cwd=None, check=True):
    print('$', ' '.join(str(x) for x in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != COMMIT:
    run(['git', 'reset', '--hard', COMMIT], cwd=REPO_DIR)
run(['uv', 'sync', '--frozen'], cwd=REPO_DIR)
run(['bash', 'tools/download_data.sh'], cwd=REPO_DIR)
print('repo_and_motion_data=READY')


In [ ]:
import shutil
import subprocess
import sys
try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import HfApi, hf_hub_download
if not HF_TOKEN:
    raise RuntimeError('缺少 HF_TOKEN')
api = HfApi(token=HF_TOKEN)
HF_REPO = f"{api.whoami()['name']}/{HF_REPO_NAME}"
for name, item in RUNS.items():
    root = item['root']
    root.mkdir(parents=True, exist_ok=True)
    target = root / 'eval_traj.npz'
    if not target.exists():
        remote = f"runs/run_{item['run_id']}/eval/eval_traj.npz"
        path = hf_hub_download(repo_id=HF_REPO, filename=remote, local_dir=str(root / 'hf_download'), token=HF_TOKEN)
        shutil.copy(path, target)
    print(f"{name}: traj={target} size={target.stat().st_size / 1e6:.1f} MB")


In [ ]:
import numpy as np
traj = np.load(str(RUNS['new_105913']['root'] / 'eval_traj.npz'))
clips = [str(x) for x in traj['names']]
assert len(clips) == 40, f'expected 40 clips, found {len(clips)}'
Path('/kaggle/working/full40_clip_manifest.txt').write_text('\n'.join(clips) + '\n')
print(f'full_clip_count={len(clips)}')
print('first=', clips[:3])
print('last=', clips[-3:])


In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
clips_arg = clips
def render_one(name, item):
    root = item['root']
    video = root / 'eval_video_all40.mp4'
    command = ['uv', 'run', 'python', 'tools/render.py', str(root / 'eval_traj.npz'), str(video), '--clips', *clips_arg, '--seconds', str(CLIP_SECONDS), '--fps', str(VIDEO_FPS), '--width', str(VIDEO_WIDTH), '--height', str(VIDEO_HEIGHT), '--title', f"BlinkMimic-T4 {item['run_id']} all 40 clips"]
    env = os.environ.copy()
    env['MUJOCO_GL'] = 'egl'
    start = time.perf_counter()
    result = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
    (root / 'eval_render_all40_stdout.txt').write_text(result.stdout + '\n' + result.stderr)
    return name, result, time.perf_counter() - start, video
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(lambda pair: render_one(*pair), RUNS.items()))
for name, result, seconds, video in results:
    print(f'{name}: exit={result.returncode} seconds={seconds:.1f} video={video} size={video.stat().st_size / 1e6:.1f} MB' if video.exists() else f'{name}: exit={result.returncode} seconds={seconds:.1f} video_missing')
    if result.returncode != 0:
        print(result.stderr[-5000:])
        raise RuntimeError(f'{name} full-40 render failed')


In [ ]:
from IPython.display import Video, display
for name, item in RUNS.items():
    root = item['root']
    for path in (root / 'eval_video_all40.mp4', root / 'eval_render_all40_stdout.txt'):
        if path.exists():
            api.upload_file(path_or_fileobj=str(path), path_in_repo=f"runs/run_{item['run_id']}/eval/{path.name}", repo_id=HF_REPO, repo_type='model')
            print(f'[hf] uploaded {name}/{path.name}')
    video = root / 'eval_video_all40.mp4'
    if video.exists(): display(Video(str(video), embed=False, width=640))
print('full40_upload=PASS')
